### Load Data

In [313]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('titanic')

print("Path to competition files:", path)

Path to competition files: C:\Users\robmc\.cache\kagglehub\competitions\titanic


In [314]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

train_data = pd.read_csv(path + '/train.csv')
test_data = pd.read_csv(path + '/test.csv')

### Tutorial - Baseline Score

In [38]:
from sklearn.ensemble import RandomForestClassifier

y = train_data["Survived"]

features = ["Pclass", "Sex", "SibSp", "Parch"]
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!


# Preparing the Data

In [315]:
train = train_data.copy()
test = test_data.copy()

train.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [316]:
train["Title"] = train["Name"].str.extract(' ([A-Za-z]+)\\.', expand=False)

train = train.drop(columns=['Name', 'Ticket', 'PassengerId'])
train.head(1)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,Title
0,0,3,male,22.0,1,0,7.25,NaN,S,Mr


In [317]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    str    
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Cabin     204 non-null    str    
 8   Embarked  889 non-null    str    
 9   Title     891 non-null    str    
dtypes: float64(2), int64(4), str(4)
memory usage: 69.7 KB


In [318]:
train['Embarked'] = train['Embarked'].fillna('S')

train['Age'] = train['Age'].fillna(train['Age'].median())

train["Deck"] = ""
for index, cabin in train['Cabin'].items():
    train.loc[index, "Deck"] = "U" if pd.isna(cabin) else cabin[0]

train = train.drop(columns=['Cabin'])

In [319]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    str    
 3   Age       891 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  891 non-null    str    
 8   Title     891 non-null    str    
 9   Deck      891 non-null    str    
dtypes: float64(2), int64(4), str(4)
memory usage: 69.7 KB


In [320]:
train['FamSize'] = train['SibSp'] + train['Parch'] + 1
train = train.drop(columns=['SibSp', 'Parch'])

In [321]:
train['Sex'] = train['Sex'].map({'female': 0, 'male': 1})

In [322]:
train['Embarked'].unique()

<StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str

In [323]:
train = pd.get_dummies(train, columns=['Embarked'])

In [324]:
train['Title'].value_counts()

Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64

In [325]:
train['Title'] = train['Title'].replace(['Ms', 'Mlle'], 'Miss')
train['Title'] = train['Title'].replace(['Sir'], 'Mr')
train['Title'] = train['Title'].replace(['Lady', 'Mme'], 'Mrs')
train['Title'] = train['Title'].replace(['Jonkheer', 'Countess', 'Don', 'Rev', 'Capt', 'Col', 'Major', 'Dr'], 'Important')

In [326]:
train['Title'].value_counts()

Title
Mr           518
Miss         185
Mrs          127
Master        40
Important     21
Name: count, dtype: int64

In [327]:
train = pd.get_dummies(train, columns=['Title'])

In [328]:
train['Deck'].value_counts()

Deck
U    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [329]:
train['Deck'] = train['Deck'].map({'G': 0, 'U': 1, 'F': 2, 'E': 3, 'D': 4, 'C': 5, 'B': 6, 'A': 7, 'T': 8})

In [330]:
train.head(2)

,Survived,Pclass,Sex,Age,Fare,Deck,FamSize,Embarked_C,Embarked_Q,Embarked_S,Title_Important,Title_Master,Title_Miss,Title_Mr,Title_Mrs
0,0,3,1,22.0,7.2500,1,2,False,False,True,False,False,False,True,False
1,1,1,0,38.0,71.2833,5,2,True,False,False,False,False,False,False,True


In [331]:
test["Title"] = test["Name"].str.extract(' ([A-Za-z]+)\\.', expand=False)

test = test.drop(columns=['Name', 'Ticket'])

test['Embarked'] = test['Embarked'].fillna('S')

test['Age'] = test['Age'].fillna(test['Age'].median())

test["Deck"] = ""
for index, cabin in test['Cabin'].items():
    test.loc[index, "Deck"] = "U" if pd.isna(cabin) else cabin[0]

test = test.drop(columns=['Cabin'])

test['FamSize'] = test['SibSp'] + test['Parch'] + 1
test = test.drop(columns=['SibSp', 'Parch'])

test['Sex'] = test['Sex'].map({'female': 0, 'male': 1})

test = pd.get_dummies(test, columns=['Embarked'])

test['Title'] = test['Title'].replace(['Ms', 'Mlle'], 'Miss')
test['Title'] = test['Title'].replace(['Sir'], 'Mr')
test['Title'] = test['Title'].replace(['Lady', 'Mme'], 'Mrs')
test['Title'] = test['Title'].replace(['Jonkheer', 'Countess', 'Don', 'Rev', 'Capt', 'Col', 'Major', 'Dr'], 'Important')
test = pd.get_dummies(test, columns=['Title'])

test['Deck'] = test['Deck'].map({'G': 0, 'U': 1, 'F': 2, 'E': 3, 'D': 4, 'C': 5, 'B': 6, 'A': 7, 'T': 8})

# Base Decision Tree

In [332]:
train_set = train.copy()
test_set = test.copy()

# Improved` Decision Tree

# Base Random Forest

# Improved Random Forest

# Future Ideas

In [36]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

% of women who survived: 0.7420382165605095
% of men who survived: 0.18890814558058924
